In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from src.preprocess import (
    load_data,
    clean_data,
    handle_missing_values
)

df = load_data("../data/customer_churn_1M.csv")

df = clean_data(df)

df = handle_missing_values(df)

In [3]:
target = "churn"

In [4]:
categorical_cols = (
    df.select_dtypes(include=["object"])
      .columns
      .tolist()
)

numerical_cols = (
    df.select_dtypes(
        include=["int64", "float64"]
    )
    .columns
    .tolist()
)

numerical_cols.remove(target)

print(categorical_cols)
print(len(numerical_cols))

['gender', 'education', 'marital_status', 'contract', 'payment_method', 'paperless_billing']
24


In [5]:
X = df.drop("churn", axis=1)

y = df["churn"]

print(X.shape)
print(y.shape)

(1000000, 30)
(1000000,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(800000, 30)
(200000, 30)


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


In [9]:
model.fit(X_train, y_train)

print("Training Complete")

Training Complete


d:\ML proj\customer_churn_ml\env\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
import joblib

joblib.dump(model, "../models/logistic_regression.pkl")
joblib.dump(X_test, "../models/X_test.pkl")
joblib.dump(y_test, "../models/y_test.pkl")

['../models/y_test.pkl']